In [19]:
#Librerias
#!pip install pandas numpy matplotlib seaborn scikit-learn imblearn xgboost tensorflow


In [20]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix
import numpy as np

data = pd.read_csv("Datasetcompleto.csv")

# Columnas irrelevantes
data = data.drop(['Source', 'Destination', 'Info', 'Label', 'Source port', 'Destination port'], axis=1)

C:\Users\User\AppData\Local\Temp\ipykernel_10552\3174369284.py:9: DtypeWarning: Columns (0: TCP Flags, 1: TCP Syn, 2: TCP ACK, 3: TCP FIN, 4: TCP RST, 5: TCP PSH, 6: TCP URG, 7: ICMP Checksum, 8: ICMP Sequence Number) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("Datasetcompleto.csv")


In [21]:
# Separar features y target
X = data.drop('Attack Category', axis=1)
y = data['Attack Category']

# Division de dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Tamaños
print("Tamaño Train:", X_train.shape)
print("Tamaño Test:", X_test.shape)

# Distribucion de clases
print("Distribución Train:\n", y_train.value_counts())
print("Distribución Test:\n", y_test.value_counts())

Tamaño Train: (51087, 15)
Tamaño Test: (21895, 15)
Distribución Train:
 Attack Category
DoS                   22994
Port scan             12636
Vulnerability scan    12435
Normal                 3022
Name: count, dtype: int64
Distribución Test:
 Attack Category
DoS                   9855
Port scan             5416
Vulnerability scan    5329
Normal                1295
Name: count, dtype: int64


In [22]:
from imblearn.combine import SMOTETomek
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Columnas
num_cols = [
    'Duration', 'Length', 'TCP Window Size', 'TCP Sequence Number'
]

tcp_flags = [
    'TCP Syn', 'TCP ACK', 'TCP FIN', 'TCP RST', 'TCP PSH', 'TCP URG',
]

cat_cols = ['Protocol', 'ICMP Type']

features = num_cols + tcp_flags + cat_cols 

# Flags TCP a 0/1
def convert_flag(val):
    return 1 if str(val).strip() in ['1', 'True', 'true'] else 0
    
for col in tcp_flags:
    X_train[col] = X_train[col].apply(convert_flag)
    X_test[col] = X_test[col].apply(convert_flag)

# Codificar columnas categóricas
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_test[col] = X_test[col].apply(lambda x: le.transform([x])[0] if x in le.classes_ else -1)
    le_dict[col] = le

# Escalamiento
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# Balanceo con SMOTETomek
print("Distribución original en y_train:\n", y_train.value_counts())

# Aplicar SMOTETomek 
smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train[features], y_train)

print("\nDistribución balanceada en y_train_res:\n", y_train_res.value_counts())
print("\nTamaño X_train_res:", X_train_res.shape)
print("Tamaño X_test:", X_test[features].shape)


Distribución original en y_train:
 Attack Category
DoS                   22994
Port scan             12636
Vulnerability scan    12435
Normal                 3022
Name: count, dtype: int64

Distribución balanceada en y_train_res:
 Attack Category
DoS                   22994
Normal                22994
Port scan             22616
Vulnerability scan    22616
Name: count, dtype: int64

Tamaño X_train_res: (91220, 12)
Tamaño X_test: (21895, 12)


In [23]:
from sklearn.ensemble import RandomForestClassifier

# Codificar target
le_attack = LabelEncoder()
y_train_res_enc = le_attack.fit_transform(y_train_res)
y_test_enc = le_attack.transform(y_test)


In [24]:
# XGBoost
xgb_model = XGBClassifier(random_state=42, n_estimators=400, max_depth=8, gamma=0.3, learning_rate=0.03, n_jobs=-1)
xgb_model.fit(X_train_res, y_train_res_enc)
y_pred_xgb = xgb_model.predict(X_test[features])

In [25]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Metricas de modelos
def mostrar_metricas(nombre_modelo, y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro')
    recall = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')

    print(f"\n Métricas Totales - {nombre_modelo}")
    print(f" Accuracy:  {accuracy:.6f}")
    print(f" Precision: {precision:.6f}")
    print(f" Recall:    {recall:.6f}")
    print(f" F1-score:  {f1:.6f}")

# Métricas para XGBoost
mostrar_metricas("XGBoost", y_test_enc, y_pred_xgb)



 Métricas Totales - XGBoost
 Accuracy:  0.948892
 Precision: 0.950487
 Recall:    0.947703
 F1-score:  0.947721


In [26]:
models = {
    "XGBoost": XGBClassifier(
gamma=0.3, learning_rate= 0.03, max_depth= 8, n_estimators= 400,
        random_state=42,
        n_jobs=-1
    )
}

cross_val_scores = {}
for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train_res, y_train_res_enc,
        scoring='recall_macro',
        cv=5,
        n_jobs=-1
    )
    cross_val_scores[name] = {
        'media': scores.mean(),
        'std': scores.std(),
        'scores': scores
    }
    print(f"   • {name:<20}: {scores.mean():.4f} (±{scores.std():.4f})")

   • XGBoost             : 0.9491 (±0.0017)


In [27]:
import joblib
joblib.dump(xgb_model, "modelo_xgboost.pkl")
joblib.dump(le_attack, "label_encoder.pkl")
joblib.dump(features, "features.pkl")
joblib.dump(le_dict, "encoders.pkl")   # 🔥 NUEVO
joblib.dump(scaler, "scaler.pkl")      # 🔥 NUEVO

['scaler.pkl']